# 🚀 Custom AI Enhancer — Kaggle Cloud GPU Training Pipeline
- Model: CodeFormer Stage III CFT Fine-Tuning with ArcFace Identity Loss
- Hardware: Kaggle Nvidia T4 / P100 GPU
- Target: 2,000 Iterations + ONNX Export

In [ ]:
import os, sys, subprocess, shutil, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    try:
        c = torch.nn.Conv2d(3, 3, 3).cuda()
        x = torch.randn(1, 3, 32, 32, device='cuda')
        _ = c(x)
        print("Native CUDA Conv2D verification PASSED!")
    except Exception as e:
        print(f"CUDA check failed ({e}). Installing compatible PyTorch...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
            'torch==2.4.0+cu121', 'torchvision==0.19.0+cu121',
            '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
        print('Compatible PyTorch installed!')
else:
    print('WARNING: Running on CPU!')


In [ ]:
# 1. Clone repository
%cd /kaggle/working
if os.path.exists('custom-ai-enhancer'):
    shutil.rmtree('custom-ai-enhancer')
!git clone https://github.com/supli6669/Enhance-Image.git custom-ai-enhancer
%cd /kaggle/working/custom-ai-enhancer

# 2. Install dependencies — pin onnxscript to a version compatible with
#    this PyTorch (the latest onnxscript dropped ParamSchema which torch.onnx needs)
!pip install -q facexlib lpips gdown onnx 'onnxscript==0.1.0.dev20231023' onnxruntime-gpu pyyaml opencv-python scikit-image
!python tools/patch_and_install_basicsr.py

# 3. Download weights and prepare dataset
!python tools/download_weights.py
!python tools/prepare_toy_training.py
print("Environment, weights, and dataset ready!")

In [ ]:
# 4. Launch GPU Training with ArcFace Identity Loss
%cd /kaggle/working/custom-ai-enhancer
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!python train_custom.py


In [ ]:
# 5. Export to ONNX & INT8 Quantization
%cd /kaggle/working/custom-ai-enhancer
import os, sys, shutil, glob

onnx_ok = False
try:
    ret = os.system("python tools/export_onnx.py")
    onnx_ok = (ret == 0) and os.path.exists("weights/CodeFormer/codeformer.onnx")
    if onnx_ok:
        print("[OK] ONNX export succeeded.")
    else:
        print("[WARN] ONNX export exited non-zero. Skipping quantization.")
except Exception as e:
    print(f"[WARN] ONNX export failed: {e}. Skipping quantization.")

if onnx_ok:
    try:
        os.system("python tools/quantize_onnx_static.py")
    except Exception as e:
        print(f"[WARN] Quantization failed: {e}")

# Copy all ONNX and weight files to /kaggle/working/ for direct download
for f in glob.glob("weights/CodeFormer/codeformer*.*"):
    dst = os.path.join("/kaggle/working", os.path.basename(f))
    shutil.copy(f, dst)
    print(f"Copied {f} -> {dst}")

# Copy the best .pth checkpoint from models/CodeFormer/experiments
pth_files = sorted(glob.glob("models/CodeFormer/experiments/**/net_g_*.pth", recursive=True) + glob.glob("experiments/**/net_g_*.pth", recursive=True))
if pth_files:
    latest = pth_files[-1]
    shutil.copy(latest, "/kaggle/working/codeformer_finetuned_v3.pth")
    print(f"Checkpoint saved: {latest} -> /kaggle/working/codeformer_finetuned_v3.pth")
else:
    print("[WARN] No .pth checkpoint found in experiments/")

print("\nTraining and Export finished! Outputs saved in /kaggle/working/")
print("Files available for download:")
for f in os.listdir("/kaggle/working"):
    fp = f"/kaggle/working/{f}"
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / 1024 / 1024
        print(f"  {f}: {size_mb:.1f} MB")
